# CNN Image Classification

This notebook trains a Convolutional Neural Network (CNN) to classify images from various datasets, then evaluates performance with metrics and visualizations.

**Supported datasets:**
- **CIFAR-10**: 32x32 color images in 10 classes (airplanes, cars, birds, etc.)
- **MNIST**: 28x28 grayscale handwritten digits (0-9)
- **Fashion-MNIST**: 28x28 grayscale clothing items (t-shirts, trousers, shoes, etc.)

The model architecture automatically adapts to the input dimensions and number of channels for each dataset.

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import wandb
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

# Import shared utilities from local package
from aiml_notebooks import (
    log_gradients, log_model_weights, log_gradient_flow,
    create_trainer, create_dataset, create_dataloaders,
    show_image_grid_normalized, plot_confusion_matrix,
    get_dataset_config,
)

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. 

PyTorch: 2.9.0
Lightning: 2.5.5


In [2]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'dataset': 'mnist',              # Dataset to use: 'cifar10', 'mnist', or 'fashionmnist'
    'seed': 42,                      # Random seed for reproducibility
    'batch_size': 128,               # Number of examples per training batch
    'num_workers': 4,                # Number of parallel data loading workers
    
    # Model
    'num_conv_layers': 3,            # Number of convolutional blocks
    'base_channels': 32,             # Number of channels in first conv layer (doubles each block)
    'dropout': 0.5,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 2,                # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Weights & Biases
    'wandb_project': 'cnn-image-classification',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Seed set to 42


42

Now we'll use the dataset factory to load the selected dataset:
- **MNIST**: 70,000 28x28 grayscale handwritten digits (60k train, 10k test)
- **Fashion-MNIST**: 70,000 28x28 grayscale clothing images in 10 classes (60k train, 10k test)
- **CIFAR-10**: 60,000 32x32 color images in 10 classes (50k train, 10k test)

In [3]:
# Get dataset configuration using shared library function
dataset_config = get_dataset_config(CONFIG['dataset'])
class_names = dataset_config['classes']
num_classes = len(class_names)
num_channels = dataset_config['num_channels']
image_size = dataset_config['image_size']

# Define data transforms (normalization and data augmentation for training)
# Training transforms with data augmentation
if CONFIG['dataset'] == 'cifar10':
    # CIFAR-10 specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(image_size, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])
elif CONFIG['dataset'] in ['mnist', 'fashionmnist']:
    # MNIST/Fashion-MNIST specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_config['mean'], dataset_config['std'])
])

# Load dataset using the factory (returns train, test)
train_dataset, test_dataset = create_dataset(
    dataset_id=CONFIG['dataset'],
    train_transform=train_transform,
    test_transform=test_transform
)

print(f"Dataset: {dataset_config['name']}")
print(f"Image size: {image_size}x{image_size}x{num_channels}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes ({num_classes}): {', '.join(class_names)}")

Dataset: MNIST
Image size: 28x28x1
Train samples: 60000
Test samples: 10000
Classes (10): 0, 1, 2, 3, 4, 5, 6, 7, 8, 9


Now we'll use the dataloader factory to create batched, shuffled loaders for training and testing.

All visualizations (sample images, confusion matrices, predictions) will be logged to W&B for monitoring.

In [ ]:
# Create data loaders using the factory
train_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=test_dataset,  # Using test set as val set for this notebook
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    use_collate_fn=False,  # Vision datasets don't need padding
    pin_memory=True
)

print(f"✓ Data loaders created (batch_size={CONFIG['batch_size']})")

Now we'll define our CNN model with convolutional blocks, batch normalization, dropout, and a classifier.

In [5]:
# Define the CNN model with convolutional blocks and fully connected classifier
class ImageClassifierCNN(L.LightningModule):
    def __init__(
        self, 
        num_classes: int = 10,
        in_channels: int = 3,
        input_size: int = 32,
        num_conv_layers: int = 3,
        base_channels: int = 32,
        dropout: float = 0.5,
        learning_rate: float = 1e-3,
    ):
        super().__init__()
        
        self.save_hyperparameters()
        
        # Build convolutional layers dynamically
        conv_layers = []
        channels = in_channels
        
        for i in range(num_conv_layers):
            out_channels = base_channels * (2 ** i)  # Double channels each layer
            
            # Convolutional block: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU -> MaxPool
            conv_layers.extend([
                nn.Conv2d(channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)  # Halves spatial dimensions
            ])
            
            channels = out_channels
        
        self.conv_layers = nn.Sequential(*conv_layers)
        
        # TODO: understand this
        # Calculate the size of the flattened features
        # Input size after N pooling layers: input_size / (2^N)
        final_spatial_size = input_size // (2 ** num_conv_layers)
        final_channels = base_channels * (2 ** (num_conv_layers - 1))
        flattened_size = final_channels * final_spatial_size * final_spatial_size
        
        # Fully connected classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
        
        # Cross-entropy loss function
        self.criterion = nn.CrossEntropyLoss()
        
        # Track predictions for evaluation
        self.test_predictions = []
        self.test_labels = []
    
    def forward(self, x):
        # Pass through convolutional layers
        features = self.conv_layers(x)
        
        # Pass through classifier
        logits = self.classifier(features)
        
        return logits
    
    def training_step(self, batch, batch_idx):
        # Forward pass to get logits for each image in batch
        x, y = batch
        logits = self(x)
        
        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits, y)
        
        # Calculate accuracy
        preds = torch.argmax(logits, dim=1) # TODO: review
        acc = (preds == y).float().mean()
        
        # Log the loss and accuracy for this batch
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        
        # Log gradients every N steps (reduce overhead)
        if batch_idx % 10 == 0:  # TODO: softcode logging frequency
            log_gradients(self, step=self.global_step)
        
        # Return the loss for this batch
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        
        # Forward pass to get logits for each image in batch
        logits = self(x)
        
        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits, y)
        
        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Log the validation loss and accuracy for this batch
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        
        # Forward pass to get logits for each image in batch
        logits = self(x)
        
        # Calculate predictions
        preds = torch.argmax(logits, dim=1)
        
        # Store predictions and labels for confusion matrix
        self.test_predictions.extend(preds.cpu().numpy())
        self.test_labels.extend(y.cpu().numpy())
        
        # Calculate accuracy
        acc = (preds == y).float().mean()
        self.log('test_acc', acc, prog_bar=True)
        
        return acc
    
    def on_train_epoch_end(self):
        # Log detailed gradient flow visualization at end of each epoch
        log_gradient_flow(self, step=self.global_step)
        log_model_weights(self, step=self.global_step)
    
    def configure_optimizers(self):
        # TODO: softcode optimizer
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        
        # Learning rate scheduler (reduce on plateau)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.5, # TODO: what is this?
            patience=5, # TODO: what is this?
            #verbose=True
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

# Initialize the model with CONFIG hyperparameters and dataset-specific parameters
model = ImageClassifierCNN(
    num_classes=num_classes,
    in_channels=num_channels,
    input_size=image_size,
    num_conv_layers=CONFIG['num_conv_layers'],
    base_channels=CONFIG['base_channels'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
)

print(f"Initialized CNN for {dataset_config['name']}")
print(f"Input: {num_channels} x {image_size} x {image_size}")
print(f"Output: {num_classes} classes")
print(f"Architecture: {CONFIG['num_conv_layers']} convolutional blocks")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Initialized CNN for MNIST
Input: 1 x 28 x 28
Output: 10 classes
Architecture: 3 convolutional blocks
Total parameters: 882,794


Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, test_loader)

# Log sample images to W&B
sample_images, sample_labels = next(iter(train_loader))
fig = plt.figure(figsize=(8, 4))
show_image_grid_normalized(
    images=sample_images[:8],
    mean=dataset_config['mean'],
    std=dataset_config['std'],
    labels=sample_labels[:8],
    class_names=class_names,
    nrows=2,
    ncols=4,
    figsize=(8, 4)
)
wandb.log({'sample_training_images': wandb.Image(plt.gcf())})
plt.close(fig)

print("✓ Training complete. View metrics and visualizations in W&B dashboard.")

Now we'll evaluate the model on the test set. The accuracy will be printed below, and all visualizations will be logged to W&B.

In [ ]:
# Test the model
model.test_predictions = []
model.test_labels = []

# Run test
test_results = trainer.test(model, test_loader, verbose=False)

# Calculate and display test accuracy
test_acc = test_results[0]['test_acc']
print(f"\n{'='*50}")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"{'='*50}\n")

# Log to W&B
wandb.log({'test_accuracy': test_acc * 100})

Now we'll create visualizations and log them to W&B: confusion matrix and sample predictions.

In [ ]:
# Create and log confusion matrix to W&B
fig = plot_confusion_matrix(
    y_true=model.test_labels,
    y_pred=model.test_predictions,
    class_names=class_names,
    figsize=(8, 7),
    cmap='Blues',
    title='Confusion Matrix',
    show=False
)
wandb.log({'confusion_matrix': wandb.Image(fig)})
plt.close(fig)

print("✓ Confusion matrix logged to W&B")

Finally, we'll log sample predictions to W&B to see how the model performs on individual images.

In [ ]:
# Log sample predictions to W&B
@torch.no_grad()
def log_predictions(model, data_loader, num_samples=16):
    model.eval()
    
    # Get a batch
    images, labels = next(iter(data_loader))
    images = images[:num_samples]
    labels = labels[:num_samples]
    
    # Get predictions
    logits = model(images.to(model.device))
    preds = torch.argmax(logits, dim=1).cpu()
    probs = F.softmax(logits, dim=1).cpu()
    confidences = probs.max(dim=1)[0]
    
    # Create figure and log to W&B
    fig = plt.figure(figsize=(10, 10))
    show_image_grid_normalized(
        images=images,
        mean=dataset_config['mean'],
        std=dataset_config['std'],
        labels=labels,
        predictions=preds,
        confidences=confidences,
        class_names=class_names,
        nrows=4,
        ncols=4,
        figsize=(10, 10)
    )
    wandb.log({'sample_predictions': wandb.Image(plt.gcf())})
    plt.close(fig)

log_predictions(model, test_loader, num_samples=16)
print("✓ Sample predictions logged to W&B")